# Exploratory Analysis

사출성형 품질 분석 팀 프로젝트의 기존 과제 노트북입니다.
팀: 락스메이, 정유진, 모티타.

**게시용 정리:** 원 분석 코드는 유지하고 파일명·데이터 경로를 정리했습니다.
이 노트북의 표와 출력은 기존 실행 기록이며, 게시 시점에 다시 실행한 결과가 아닙니다.
대표 그림은 `results/figures/`로 분리했습니다. 원본에 없던 실험 결과를 추가하지 않았습니다.

**라벨:** 최종 모델에서 `1=정상(Y)`, `0=불량(N)`입니다. 기본 precision/recall/F1은 정상 클래스 기준입니다.
문서에 남아 있는 defect/positive 설명은 이 라벨 정의와 함께 읽어야 합니다.
성능 해석과 전처리 한계는 [실험 기록](../docs/experiment-notes.md)을 확인하세요.

저장소 루트에서 `jupyter lab`을 실행하고 이 파일을 열어 순서대로 실행하세요.


In [ ]:
from pathlib import Path
Path("../outputs").mkdir(exist_ok=True)



# Exploratory Analysis of Injection Molding Data

**Dataset:** KAMP Injection Molding AI Dataset (labeled & unlabeled variants)  
**Team 4:** 락스메이, 정유진, 모티타  
**Course:** data analytics for electronic manufacturing  / 2nd Semester

> **How to use this notebook**
> 1. Place the CSVs in the same folder as this notebook.  
> 2. Update the file paths in the **Data Loading** cell, if needed.  
> 3. Run the notebook top-to-bottom. All plots are created with `matplotlib` (no seaborn) as required.
> 4. Every figure/table is followed by a short analysis remark as required.



## 1. Business Understanding

### 1.1 Background & Process (Injection Molding)
- **Injection Molding Overview**:Injection molding is a high-throughput thermoplastic manufacturing process consisting of three main phases— **filling, pressurization**, and **compensation/holding**. The polymer granules are dried and fed into the hopper, melted in the barrel, injected into the mold cavity under controlled temperature and pressure, and finally ejected after cooling. This cycle repeats rapidly in production  
- **Typical Defects**:Common quality issues include short-shot, flash, sink marks, flow marks, weld lines, bubbles, and warpage, often arising from inappropriate process parameter combinations.
- **Key Controllable Factors**:Major process parameters influencing product quality include temperatures (barrel, hopper, mold), pressures (injection, transfer, back pressure), speeds (injection speed, screw RPM), and timings (filling, holding, cooling, cycle time)
- **Data Context**:The dataset originates from an automotive windshield side molding injection line, collected at 0.2-second intervals between October 16 and November 19, 2020, under the KAMP initiative (Ministry of SMEs and Startups & KAIST)

### 1.2 Problem Definition
- **Objective**:Predict or monitor **Pass/Fail** (good vs defective) outcomes using process sensor data, enabling early detection of anomalies and identification of root causes for quality drift.
-**Field Pain Points**:
 1. Defects are typically detected only after cooling, delaying corrective action.
 2. Complex, interacting parameters make root-cause diagnosis difficult. 3.Parameter tuning remains highly experience-dependent.
-**Scope**:The initial focus is on early warning and cause identification through real-time or batch monitoring. Long-term, insights may inform recipe optimization for process parameters (e.g., speed, temperature, pressure).

### 1.3 Stakeholders
- **Operators** & **process engineers** - Real time parameter tuning and process stability.
- **Quality teams** - Defect inspection, classification, and triage.
- **Managers** - Scrap/cost control, throughput, and KPI tracking.

### 1.4 Analysis Purpose
- Build foundational **data understanding**: schema, types, ranges, missingness, simple correlations.
- Identify early **data issues**: duplicates, anomalies (e.g., zeros where impossible), potential leakage fields.
- Produce **actionable insights** to guide the next modeling phase.



## 2. Data Understanding
### 2.1 Load Data and Describe
**Purpose**: Establish a clear understanding of schema, variable types, and structure.

### Describe:

#### The dataset contains about 7996 production records collected from an injection molding process.

- Each record represent one production cycle, including machine sensor data, operating parameters and production inspection results. After cleaning and removing inactive or redundant columns only 33 useful features remain for analysis.
- There are 3 type of values in the dataset.
  - DateTime: TimeStamp which i convert the original data from a non-null data to datetime
  - Categorical (object): PART_FACT_SERIAL, PART_NAME, EQUIP_CD, EQUIP_NAME, PassOrFail, Reason
  - Numerical (float64 & Int64): The rest of the dataset.
- Every single data features is filled with data point but from the data.info() we can see that only 71 non-null was shown in the Reasons. This indicate that only 71 cases have a fail reasons and those reasons have '가스' '미성형' '초기허용불량'
- Our main target variable is the PassOrFail which contain only "Y" for Pass and "N" for Fail. It is a binary classification which process condition correlate to the Reason.

#### Unit Type measurement

- Timing Parameters: `Units (approx)`: seconds(s) -> Time it take in each production step
- Position Parameters `Units (approx)`: millimeter (mm) -> Linear position of moving component
- Speed & RPM `Units (approx)`: revolutions per minute (RPM) -> Rotational or linear movement speed
- Pressure Parameters `Units (approx)`: bar/MPa -> System Pressures in different injection stages
- Temperature Parameters`Units (approx)`: Thermal zones of barrel hopper and mold

#### Numerical Features Range

- Timing: `Range`: 10-67.0s `Mean`: 59.8s (Cycle_Time)
- Position `Range`: 0-655 `Mean`: 653-655
- Pressure `Range`: 100-180 `Mean`: varies by stage
- Temperature `Range`: 220-280°C(barrel) 22-26°C(mold)

The dataset is mostly clean, numeric, and well-structured. It captures key physical parameters of the injection molding process, making it ideal for exploratory data analysis and later predictive modeling.



In [ ]:

import pandas as pd
import numpy as np

#Load Dataset
label_data = pd.read_csv('../data/labeled_data.csv')
label_data.head()

#Dataset Overview
print("Shape:", label_data.shape)
print("\nData Types:\n", label_data.dtypes)
print("\nMissing Ratio:\n", label_data.isnull().mean().round(3) * 100)

## Frequency and percentage
print(label_data['PassOrFail'].value_counts())
print(label_data['PassOrFail'].value_counts(normalize=True)*100)



Shape: (7996, 45)

Data Types:
 _id                          object
TimeStamp                    object
PART_FACT_PLAN_DATE          object
PART_FACT_SERIAL              int64
PART_NAME                    object
EQUIP_CD                     object
EQUIP_NAME                   object
PassOrFail                   object
Reason                       object
Injection_Time              float64
Filling_Time                float64
Plasticizing_Time           float64
Cycle_Time                  float64
Clamp_Close_Time            float64
Cushion_Position            float64
Switch_Over_Position        float64
Plasticizing_Position       float64
Clamp_Open_Position         float64
Max_Injection_Speed         float64
Max_Screw_RPM               float64
Average_Screw_RPM           float64
Max_Injection_Pressure      float64
Max_Switch_Over_Pressure    float64
Max_Back_Pressure           float64
Average_Back_Pressure       float64
Barrel_Temperature_1        float64
Barrel_Temperature_2        floa

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
plt.rcParams["figure.figsize"] = (6,4); plt.rcParams["axes.grid"] = True
df = label_data
target_col = next((c for c in ["PassOrFail","passorfail","label","Label","target"] if c in df.columns), None)
product_col = next((c for c in ["Product_Code","PRODUCT_CODE","product","product_code"] if c in df.columns), None)
equip_col = next((c for c in ["EQUIP_NAME","EQUIP_CD","EQUIP_ID","Machine","Equip"] if c in df.columns), None)
print("Detected target:", target_col); print("Detected Product_Code:", product_col); print("Detected equipment:", equip_col)

df.head()

Detected target: PassOrFail
Detected Product_Code: None
Detected equipment: EQUIP_NAME


                        _id            TimeStamp     PART_FACT_PLAN_DATE  \
0  5f8928bb9c0189cc666ef19b  2020-10-16 04:57:47  2020-10-16 오전 12:00:00   
1  5f8928de9c0189cc666ef20b  2020-10-16 04:58:48  2020-10-16 오전 12:00:00   
2  5f8928df9c0189cc666ef213  2020-10-16 04:58:48  2020-10-16 오전 12:00:00   
3  5f8928f39c0189cc666ef25e  2020-10-16 04:59:48  2020-10-16 오전 12:00:00   
4  5f8928f59c0189cc666ef265  2020-10-16 04:59:48  2020-10-16 오전 12:00:00   

   PART_FACT_SERIAL              PART_NAME EQUIP_CD  EQUIP_NAME PassOrFail  \
0                24  CN7 W/S SIDE MLD'G RH      S14  650톤-우진2호기          Y   
1                24  CN7 W/S SIDE MLD'G RH      S14  650톤-우진2호기          Y   
2                23  CN7 W/S SIDE MLD'G LH      S14  650톤-우진2호기          Y   
3                23  CN7 W/S SIDE MLD'G LH      S14  650톤-우진2호기          Y   
4                24  CN7 W/S SIDE MLD'G RH      S14  650톤-우진2호기          Y   

  Reason  Injection_Time  ...  Mold_Temperature_3  Mold_Temperature_4  \
0


### 2.2 Feature Dictionary (Types • Units • Notes)
Below is a concise dictionary for the main variables used in EDA. Adjust units/types if your specific file differs.


In [ ]:
# Extract type information
feature_dict = pd.DataFrame({
    'Feature': label_data.columns,
    'Type': label_data.dtypes.astype(str)
})

# Optionally define units & notes manually for main groups
unit_map = {
    'Time': 'sec',
    'Pressure': 'bar',
    'Speed': 'mm/s',
    'Temperature': '°C',
    'Position': 'mm'
}

feature_dict.head(15)

                                 Feature     Type
_id                                  _id   object
TimeStamp                      TimeStamp   object
PART_FACT_PLAN_DATE  PART_FACT_PLAN_DATE   object
PART_FACT_SERIAL        PART_FACT_SERIAL    int64
PART_NAME                      PART_NAME   object
EQUIP_CD                        EQUIP_CD   object
EQUIP_NAME                    EQUIP_NAME   object
PassOrFail                    PassOrFail   object
Reason                            Reason   object
Injection_Time            Injection_Time  float64
Filling_Time                Filling_Time  float64
Plasticizing_Time      Plasticizing_Time  float64
Cycle_Time                    Cycle_Time  float64
Clamp_Close_Time        Clamp_Close_Time  float64
Cushion_Position        Cushion_Position  float64

In [ ]:
# Describe main features
import pandas as pd
data_dict = pd.DataFrame([
    ("Injection_Time", "float", "sec", "High-pressure injection duration"),
    ("Filling_Time", "float", "sec", "Time to fill mold cavity"),
    ("Plasticizing_Time", "float", "sec", "Time to melt/prepare material for next shot"),
    ("Cycle_Time", "float", "sec", "Total duration of one production cycle"),
    ("Clamp_Close_Time", "float", "sec", "Time to close and clamp mold"),
    ("Cushion_Position", "float", "mm", "Screw position for holding pressure"),
    ("Switch_Over_Position", "float", "mm", "Position when switching to holding pressure"),
    ("Plasticizing_Position", "float", "mm", "Screw position at end of plasticizing"),
    ("Clamp_Open_Position", "float", "mm", "Position when mold opens for ejection"),
    ("Max_Injection_Speed", "float", "mm/s", "Max speed of injected melt"),
    ("Max_Screw_RPM", "float", "mm/s", "Max screw rotation speed"),
    ("Average_Screw_RPM", "float", "mm/s", "Average screw rotation speed"),
    ("Max_Injection_Pressure", "float", "MPa", "Maximum nozzle/cavity pressure"),
    ("Max_Switch_Over_Pressure", "float", "MPa", "Pressure at switchover"),
    ("Max_Back_Pressure", "float", "MPa", "Max back pressure during plasticizing"),
    ("Average_Back_Pressure", "float", "MPa", "Average back pressure"),
    ("Barrel_Temperature_1", "float", "°C", "Barrel zone temperature 1"),
    ("Barrel_Temperature_2", "float", "°C", "Barrel zone temperature 2"),
    ("Barrel_Temperature_3", "float", "°C", "Barrel zone temperature 3"),
    ("Barrel_Temperature_4", "float", "°C", "Barrel zone temperature 4"),
    ("Barrel_Temperature_5", "float", "°C", "Barrel zone temperature 5"),
    ("Barrel_Temperature_6", "float", "°C", "Barrel zone temperature 6"),
    ("Hopper_Temperature", "float", "°C", "Hopper temperature"),
    ("Mold_Temperature_3", "float", "°C", "Mold temperature zone 3"),
    ("Mold_Temperature_4", "float", "°C", "Mold temperature zone 4"),
    ("PassOrFail", "int (0/1)", "-", "Target: 1=good, 0=defect (labeled only)"),
], columns=["Feature","Type","Unit","Description"])

data_dict


                     Feature       Type  Unit  \
0             Injection_Time      float   sec   
1               Filling_Time      float   sec   
2          Plasticizing_Time      float   sec   
3                 Cycle_Time      float   sec   
4           Clamp_Close_Time      float   sec   
5           Cushion_Position      float    mm   
6       Switch_Over_Position      float    mm   
7      Plasticizing_Position      float    mm   
8        Clamp_Open_Position      float    mm   
9        Max_Injection_Speed      float  mm/s   
10             Max_Screw_RPM      float  mm/s   
11         Average_Screw_RPM      float  mm/s   
12    Max_Injection_Pressure      float   MPa   
13  Max_Switch_Over_Pressure      float   MPa   
14         Max_Back_Pressure      float   MPa   
15     Average_Back_Pressure      float   MPa   
16      Barrel_Temperature_1      float    °C   
17      Barrel_Temperature_2      float    °C   
18      Barrel_Temperature_3      float    °C   
19      Barrel_Tempe


### 2.3 Structure, Missingness, and Summary Stats
We review dtypes, missing percentages, and descriptive statistics.


In [ ]:
# dtypes & missingness summary
dtypes = label_data.dtypes.to_frame("dtype")
missing_pct = label_data.isna().mean().to_frame("missing_pct").round(4)

schema = dtypes.join(missing_pct, how="left")
schema.sort_index()


                            dtype  missing_pct
Average_Back_Pressure     float64       0.0000
Average_Screw_RPM         float64       0.0000
Barrel_Temperature_1      float64       0.0000
Barrel_Temperature_2      float64       0.0000
Barrel_Temperature_3      float64       0.0000
Barrel_Temperature_4      float64       0.0000
Barrel_Temperature_5      float64       0.0000
Barrel_Temperature_6      float64       0.0000
Barrel_Temperature_7      float64       0.0000
Clamp_Close_Time          float64       0.0000
Clamp_Open_Position       float64       0.0000
Cushion_Position          float64       0.0000
Cycle_Time                float64       0.0000
EQUIP_CD                   object       0.0000
EQUIP_NAME                 object       0.0000
Filling_Time              float64       0.0000
Hopper_Temperature        float64       0.0000
Injection_Time            float64       0.0000
Max_Back_Pressure         float64       0.0000
Max_Injection_Pressure    float64       0.0000
Max_Injection

In [ ]:
summary = label_data.describe(include='all').T
display(summary.head(10))


                      count unique                       top  freq       mean  \
_id                    7996   5232  5fa112bc9c0189cc66dac23a     2        NaN   
TimeStamp              7996   2625       2020-11-03 00:59:02     6        NaN   
PART_FACT_PLAN_DATE    7996     13    2020-11-03 오전 12:00:00  1854        NaN   
PART_FACT_SERIAL     7996.0    NaN                       NaN   NaN  11.248749   
PART_NAME              7996      6     CN7 W/S SIDE MLD'G RH  3371        NaN   
EQUIP_CD               7996      3                       S14  7992        NaN   
EQUIP_NAME             7996      3                650톤-우진2호기  7992        NaN   
PassOrFail             7996      2                         Y  7925        NaN   
Reason                   71      3                        가스    35        NaN   
Injection_Time       7996.0    NaN                       NaN   NaN   8.242189   

                          std   min   25%   50%   75%        max  
_id                       NaN   NaN   NaN


**Analysis remark:**  
- Check any features with **>30% missing** or constant values (variance ~0). Consider dropping or imputing before modeling.



### 2.4 Early Data Issues & Assumptions

Because dataset, which merges several product types and machine lines, a few important data-quality and structural issues naturally appear.
Below is a list of the main issues and their recommended solutions.
- The dataset may mix several **machine IDs and product molds**, so grouping by equipment or product code is recommended before training.
- **Duplicates:** check row-wise duplicates and remove if necessary.  

- Some **sensors are constant** (value = 0 or same across cycles) → remove such columns.  
- **Label leakage risk:** columns derived from inspection systems or manual checks must be excluded when modeling.  
- Data were collected over different days; time drift (e.g., temperature variance) may exist → verify temporal consistency.


In [ ]:
df = label_data.copy()
# Duplicates
dup_count = df.duplicated().sum()
print(f"Duplicate rows: {dup_count}")

# Simple anomaly flags
issue_report = pd.DataFrame({
    "neg_times": (df[['Injection_Time','Filling_Time','Plasticizing_Time','Cycle_Time']].lt(0)).any(axis=1),
    "zero_mold_temps": (df.filter(like='Mold_Temperature_').eq(0)).any(axis=1) if df.filter(like='Mold_Temperature_').shape[1] > 0 else False,
    "zero_barrel_t1": df['Barrel_Temperature_1'].eq(0) if 'Barrel_Temperature_1' in df else False
})

summary = issue_report.value_counts().to_frame("count").sort_values("count", ascending=False)
display(summary.head(10))


Duplicate rows: 2764


                                          count
neg_times zero_mold_temps zero_barrel_t1       
False     True            False            7994
                          True                2


**Analysis remark:**  
- Presence of **zero mold temperatures** or zeros in key thermals often indicates missing sensors or data acquisition gaps; treat as missing or filtered out.



## 3. Exploratory Data Analysis (EDA)

> All plots use **matplotlib** (no seaborn), one figure per chart, as requested.


### 3.1 Histograms

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots()
df['Cycle_Time'].dropna().plot(kind='hist', bins=30, ax=ax, title='Histogram: Cycle_Time')
plt.xlabel('Cycle_Time (sec)')
plt.ylabel('Count')
plt.show()


[Historical chart exported to results/figures/cycle-time-distribution.png]


**Insight:** Cycle_Time appears slightly right-skewed with occasional long cycles, hinting at sporadic slowdowns that may relate to defects or rework.


### 3.2 Boxplot: Injection_Time by PassOrFail

In [ ]:
import matplotlib.pyplot as plt
if 'PassOrFail' in df.columns and df['PassOrFail'].notna().any():
    fig, ax = plt.subplots()
    df.boxplot(column='Injection_Time', by='PassOrFail', ax=ax)
    plt.title('Injection_Time by PassOrFail')
    plt.suptitle('')
    plt.xlabel('PassOrFail (1=good, 0=defect)')
    plt.ylabel('Injection_Time (sec)')
    plt.show()
else:
    print("Target not available; skipping boxplot.")


[Historical plot output omitted; rerun this cell to display it.]


**Insight:** If defects show longer Injection_Time, it may indicate overpacking or difficult filling; investigate switchover timing and pressure ramps.


### 3.3 Violin Plot: Max_Injection_Pressure by Pass/Fail

In [ ]:
plt.figure()
data_passfail = [df.loc[df['PassOrFail'] == val, 'Max_Injection_Pressure'] for val in df['PassOrFail'].unique()]
plt.violinplot(data_passfail, showmeans=True)
plt.title('Violin Plot: Max_Injection_Pressure by Pass/Fail')
plt.xlabel('1=Pass, 2=Fail')
plt.ylabel('Max_Injection_Pressure (bar)')
plt.show()


[Historical plot output omitted; rerun this cell to display it.]


**Insight:** Strong positive correlation among barrel temperature zones is expected; watch for multi-collinearity before modeling.


### 3.4 Bar Chart: Defect Reasons (if present)

In [ ]:
import matplotlib.pyplot as plt
base = pd.read_csv("../data/labeled_data.csv")

if 'Reason' in base.columns:
    counts = base['Reason'].value_counts(dropna=True).head(15)
    fig, ax = plt.subplots()
    counts.plot(kind='bar', ax=ax, title='Top Defect Reasons')
    plt.xlabel('Reason')
    plt.ylabel('Count')
    plt.tight_layout()
    plt.show()
else:
    print("Column 'Reason' not found; skipping.")


/tmp/ipython-input-520488842.py:10: UserWarning: Glyph 44032 (\N{HANGUL SYLLABLE GA}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipython-input-520488842.py:10: UserWarning: Glyph 49828 (\N{HANGUL SYLLABLE SEU}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipython-input-520488842.py:10: UserWarning: Glyph 52488 (\N{HANGUL SYLLABLE CO}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipython-input-520488842.py:10: UserWarning: Glyph 44592 (\N{HANGUL SYLLABLE GI}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipython-input-520488842.py:10: UserWarning: Glyph 54728 (\N{HANGUL SYLLABLE HEO}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipython-input-520488842.py:10: UserWarning: Glyph 50857 (\N{HANGUL SYLLABLE YONG}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipython-input-520488842.py:10: UserWarning: Glyph 48520 (\N{HANGUL SYLLABLE BUL}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipy

[Historical plot output omitted; rerun this cell to display it.]


**Insight:** Concentration in a few defect categories suggests targeted countermeasures (e.g., tune temperatures for flow-related issues).


### 3.5 Line Plot: Average Barrel Temperature per Zone

In [ ]:
import matplotlib.pyplot as plt

plt.figure()
barrel_cols = [c for c in df.columns if 'Barrel_Temperature_' in c]
if barrel_cols:
    means = df[barrel_cols].mean()
    plt.plot(barrel_cols, means, marker='o')
    plt.title('Line Plot: Average Barrel Temperatures per Zone')
    plt.xlabel('Barrel Zone')
    plt.ylabel('Average Temperature (°C)')
    plt.grid(True)
    plt.show()

[Historical plot output omitted; rerun this cell to display it.]


**Insight:** Differences in pressure distribution by Pass/Fail indicate a potential operating window for stable quality.


### 3.6 Groupby / Pivot Table / Value Counts / Describe

In [ ]:
# value_counts (target rate)
if 'PassOrFail' in df.columns:
    print("Target distribution:")
    print(df['PassOrFail'].value_counts(dropna=True))
    print()

# groupby: target-wise means
numeric_cols = ['Cycle_Time','Injection_Time','Max_Injection_Pressure','Hopper_Temperature']
existing_cols = [c for c in numeric_cols if c in df.columns]

if 'PassOrFail' in df.columns and existing_cols:
    gb = df.groupby('PassOrFail')[existing_cols].mean().round(2)
    print("Groupby means by PassOrFail:")
    display(gb)

# pivot_table: average by binned cycle time vs target
cats = pd.cut(df['Cycle_Time'], bins=5, include_lowest=True)
pivot = pd.pivot_table(df.assign(CycleBin=cats), index='CycleBin', values='Max_Injection_Pressure', columns='PassOrFail', aggfunc='mean')
pivot = pivot.round(2)
pivot

#Describe summary for key numeric features
desc = df[existing_cols].describe().T
desc['range'] = desc['max'] - desc['min']
print("\nSummary statistics for key features:")
display(desc.round(2))

Target distribution:
PassOrFail
Y    7925
N      71
Name: count, dtype: int64

Groupby means by PassOrFail:


            Cycle_Time  Injection_Time  Max_Injection_Pressure  \
PassOrFail                                                       
N                60.88            6.15                  143.00   
Y                59.89            8.26                  142.18   

            Hopper_Temperature  
PassOrFail                      
N                        65.49  
Y                        66.47  


Summary statistics for key features:


/tmp/ipython-input-227474016.py:18: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  pivot = pd.pivot_table(df.assign(CycleBin=cats), index='CycleBin', values='Max_Injection_Pressure', columns='PassOrFail', aggfunc='mean')


                         count    mean   std     min     25%     50%     75%  \
Cycle_Time              7996.0   59.90  0.89   58.84   59.52   59.52   59.56   
Injection_Time          7996.0    8.24  3.11    1.05    9.52    9.56    9.59   
Max_Injection_Pressure  7996.0  142.19  1.95  134.80  141.80  142.00  142.20   
Hopper_Temperature      7996.0   66.46  2.58    0.00   65.40   66.70   67.60   

                           max  range  
Cycle_Time               66.86   8.02  
Injection_Time           16.31  15.26  
Max_Injection_Pressure  183.20  48.40  
Hopper_Temperature       70.60  70.60  

# 3.7  Distribution of Cycle Time

In [ ]:
import seaborn as sns
plt.style.use('seaborn-v0_8-darkgrid')
sns.set(font_scale=1.1)

# 1. Distribution of Cycle Time
plt.figure(figsize=(8,5))
sns.histplot(df['Cycle_Time'], kde=True, color='steelblue', bins=25)
plt.title('Distribution of Cycle Time')
plt.xlabel('Cycle Time (s)')
plt.ylabel('Frequency')
plt.show()

[Historical plot output omitted; rerun this cell to display it.]

# 3.8 Grouped EDA by `Product Type`

**Why group?** Each product operates under different nominal ranges (cycle time, pressure, mold temperature). If we mix them, patterns blur and models misinterpret product differences as noise.

**Plan per product:** target ratio → distributions (hist/box/violin) → missing % & IQR outliers → pivot (machine × target) → short takeaways.

In [ ]:
plt.rcParams["figure.figsize"] = (6,4)
plt.rcParams["axes.grid"] = True


# Detect key columns
target_col = next((c for c in ["PassOrFail","passorfail","label","Label","target"] if c in df.columns), None)
equip_col = next((c for c in ["EQUIP_NAME","EQUIP_CD","EQUIP_ID","Machine","Equip"] if c in df.columns), None)

print("Detected target:", target_col)
print("Detected equipment key:", equip_col)

if equip_col is None:
    raise ValueError("No EQUIP_NAME-like column found; cannot run grouped EDA with 10 visualizations.")

# Choose representative numeric metrics
preferred = ["Cycle_Time","Injection_Time","Filling_Time","Max_Injection_Pressure","Max_Injection_Speed"]
candidate_metrics = [c for c in preferred if c in df.columns]
if not candidate_metrics:
    nums = df.select_dtypes(include=[np.number]).columns.tolist()
    candidate_metrics = [c for c in nums if c != target_col][:5]

metricA = candidate_metrics[0]
metricB = candidate_metrics[1] if len(candidate_metrics)>1 else metricA
metricC = candidate_metrics[2] if len(candidate_metrics)>2 else metricA
metricD = candidate_metrics[3] if len(candidate_metrics)>3 else metricB

print("Metrics used:", metricA, metricB, metricC, metricD)

# Focus on top 3 machines by count
top_equips = df[equip_col].value_counts().head(3).index.tolist()
print("Top equipment IDs:", top_equips)

def to_num(s):
    return pd.to_numeric(s, errors="coerce")


Detected target: PassOrFail
Detected equipment key: EQUIP_NAME
Metrics used: Cycle_Time Injection_Time Filling_Time Max_Injection_Pressure
Top equipment IDs: ['650톤-우진2호기', '1800TON-우진', '650톤-우진']


In [ ]:
# V1) Bar chart — Pass/Fail ratio by equipment
if target_col:
    counts = df.loc[df[equip_col].isin(top_equips), [equip_col, target_col]].groupby([equip_col, target_col]).size().unstack(fill_value=0)
    counts.plot(kind="bar", rot=0)
    plt.title(f"[V1] Target distribution by {equip_col} (top 3)")
    plt.xlabel(equip_col); plt.ylabel("Count")
    plt.tight_layout(); plt.show()
    print("Remark: Machines differ in Pass/Fail ratios, hinting at hardware or calibration effects.")

/tmp/ipython-input-1965327159.py:7: UserWarning: Glyph 50864 (\N{HANGUL SYLLABLE U}) missing from font(s) DejaVu Sans.
  plt.tight_layout(); plt.show()
/tmp/ipython-input-1965327159.py:7: UserWarning: Glyph 51652 (\N{HANGUL SYLLABLE JIN}) missing from font(s) DejaVu Sans.
  plt.tight_layout(); plt.show()
/tmp/ipython-input-1965327159.py:7: UserWarning: Glyph 53668 (\N{HANGUL SYLLABLE TON}) missing from font(s) DejaVu Sans.
  plt.tight_layout(); plt.show()
/tmp/ipython-input-1965327159.py:7: UserWarning: Glyph 54840 (\N{HANGUL SYLLABLE HO}) missing from font(s) DejaVu Sans.
  plt.tight_layout(); plt.show()
/tmp/ipython-input-1965327159.py:7: UserWarning: Glyph 44592 (\N{HANGUL SYLLABLE GI}) missing from font(s) DejaVu Sans.
  plt.tight_layout(); plt.show()
/usr/local/lib/python3.12/dist-packages/IPython/core/pylabtools.py:151: UserWarning: Glyph 50864 (\N{HANGUL SYLLABLE U}) missing from font(s) DejaVu Sans.
  fig.canvas.print_figure(bytes_io, **kw)
/usr/local/lib/python3.12/dist-packag

[Historical chart exported to results/figures/quality-by-equipment.png]

Remark: Machines differ in Pass/Fail ratios, hinting at hardware or calibration effects.


In [ ]:
# V2–V4) Histograms — metricA per equipment
for i, m in enumerate(top_equips, start=2):
    s = to_num(df.loc[df[equip_col]==m, metricA]).dropna()
    if len(s)==0: continue
    s.plot(kind="hist", bins=40)
    plt.title(f"[V{i}] Histogram of {metricA} — {equip_col}={m}")
    plt.xlabel(metricA); plt.ylabel("Frequency")
    plt.tight_layout(); plt.show()
    print(f"Remark: {metricA} range for {m} reveals machine-specific process conditions.")

/tmp/ipython-input-3815962133.py:8: UserWarning: Glyph 53668 (\N{HANGUL SYLLABLE TON}) missing from font(s) DejaVu Sans.
  plt.tight_layout(); plt.show()
/tmp/ipython-input-3815962133.py:8: UserWarning: Glyph 50864 (\N{HANGUL SYLLABLE U}) missing from font(s) DejaVu Sans.
  plt.tight_layout(); plt.show()
/tmp/ipython-input-3815962133.py:8: UserWarning: Glyph 51652 (\N{HANGUL SYLLABLE JIN}) missing from font(s) DejaVu Sans.
  plt.tight_layout(); plt.show()
/tmp/ipython-input-3815962133.py:8: UserWarning: Glyph 54840 (\N{HANGUL SYLLABLE HO}) missing from font(s) DejaVu Sans.
  plt.tight_layout(); plt.show()
/tmp/ipython-input-3815962133.py:8: UserWarning: Glyph 44592 (\N{HANGUL SYLLABLE GI}) missing from font(s) DejaVu Sans.
  plt.tight_layout(); plt.show()
/usr/local/lib/python3.12/dist-packages/IPython/core/pylabtools.py:151: UserWarning: Glyph 53668 (\N{HANGUL SYLLABLE TON}) missing from font(s) DejaVu Sans.
  fig.canvas.print_figure(bytes_io, **kw)
/usr/local/lib/python3.12/dist-pack

[Historical plot output omitted; rerun this cell to display it.]

Remark: Cycle_Time range for 650톤-우진2호기 reveals machine-specific process conditions.


/tmp/ipython-input-3815962133.py:8: UserWarning: Glyph 50864 (\N{HANGUL SYLLABLE U}) missing from font(s) DejaVu Sans.
  plt.tight_layout(); plt.show()
/tmp/ipython-input-3815962133.py:8: UserWarning: Glyph 51652 (\N{HANGUL SYLLABLE JIN}) missing from font(s) DejaVu Sans.
  plt.tight_layout(); plt.show()
/usr/local/lib/python3.12/dist-packages/IPython/core/pylabtools.py:151: UserWarning: Glyph 50864 (\N{HANGUL SYLLABLE U}) missing from font(s) DejaVu Sans.
  fig.canvas.print_figure(bytes_io, **kw)
/usr/local/lib/python3.12/dist-packages/IPython/core/pylabtools.py:151: UserWarning: Glyph 51652 (\N{HANGUL SYLLABLE JIN}) missing from font(s) DejaVu Sans.
  fig.canvas.print_figure(bytes_io, **kw)


[Historical plot output omitted; rerun this cell to display it.]

Remark: Cycle_Time range for 1800TON-우진 reveals machine-specific process conditions.


/tmp/ipython-input-3815962133.py:8: UserWarning: Glyph 53668 (\N{HANGUL SYLLABLE TON}) missing from font(s) DejaVu Sans.
  plt.tight_layout(); plt.show()
/tmp/ipython-input-3815962133.py:8: UserWarning: Glyph 50864 (\N{HANGUL SYLLABLE U}) missing from font(s) DejaVu Sans.
  plt.tight_layout(); plt.show()
/tmp/ipython-input-3815962133.py:8: UserWarning: Glyph 51652 (\N{HANGUL SYLLABLE JIN}) missing from font(s) DejaVu Sans.
  plt.tight_layout(); plt.show()
/usr/local/lib/python3.12/dist-packages/IPython/core/pylabtools.py:151: UserWarning: Glyph 53668 (\N{HANGUL SYLLABLE TON}) missing from font(s) DejaVu Sans.
  fig.canvas.print_figure(bytes_io, **kw)
/usr/local/lib/python3.12/dist-packages/IPython/core/pylabtools.py:151: UserWarning: Glyph 50864 (\N{HANGUL SYLLABLE U}) missing from font(s) DejaVu Sans.
  fig.canvas.print_figure(bytes_io, **kw)
/usr/local/lib/python3.12/dist-packages/IPython/core/pylabtools.py:151: UserWarning: Glyph 51652 (\N{HANGUL SYLLABLE JIN}) missing from font(s) 

[Historical plot output omitted; rerun this cell to display it.]

Remark: Cycle_Time range for 650톤-우진 reveals machine-specific process conditions.


In [ ]:
# V5) Boxplot — metricA across equipment
data = [to_num(df.loc[df[equip_col]==m, metricA]).dropna().values for m in top_equips]
labels = [str(m) for m in top_equips]
if any(len(d)>0 for d in data) and len(labels)>=2:
    plt.boxplot(data, labels=labels)
    plt.title(f"[V5] Boxplot of {metricA} by {equip_col} (top 3)")
    plt.tight_layout(); plt.show()
    print("Remark: Median and IQR variation shows machine-to-machine differences.")


/tmp/ipython-input-1408644931.py:5: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  plt.boxplot(data, labels=labels)
/tmp/ipython-input-1408644931.py:7: UserWarning: Glyph 53668 (\N{HANGUL SYLLABLE TON}) missing from font(s) DejaVu Sans.
  plt.tight_layout(); plt.show()
/tmp/ipython-input-1408644931.py:7: UserWarning: Glyph 50864 (\N{HANGUL SYLLABLE U}) missing from font(s) DejaVu Sans.
  plt.tight_layout(); plt.show()
/tmp/ipython-input-1408644931.py:7: UserWarning: Glyph 51652 (\N{HANGUL SYLLABLE JIN}) missing from font(s) DejaVu Sans.
  plt.tight_layout(); plt.show()
/tmp/ipython-input-1408644931.py:7: UserWarning: Glyph 54840 (\N{HANGUL SYLLABLE HO}) missing from font(s) DejaVu Sans.
  plt.tight_layout(); plt.show()
/tmp/ipython-input-1408644931.py:7: UserWarning: Glyph 44592 (\N{HANGUL SYLLABLE GI}) missing from font(s) DejaVu Sans.
  plt.tight_layout(); plt.s

[Historical plot output omitted; rerun this cell to display it.]

Remark: Median and IQR variation shows machine-to-machine differences.


In [ ]:
# V6) Violin — metricB across equipment
data = [to_num(df.loc[df[equip_col]==m, metricB]).dropna().values for m in top_equips]
labels = [str(m) for m in top_equips]
if any(len(d)>0 for d in data) and len(labels)>=2:
    plt.violinplot(data, showmeans=True, showmedians=True)
    plt.xticks(range(1, len(labels)+1), labels)
    plt.title(f"[V6] Violin of {metricB} by {equip_col} (top 3)")
    plt.tight_layout(); plt.show()
    print("Remark: Density and spread differ between machines — key for quality optimization.")


/tmp/ipython-input-1249101656.py:8: UserWarning: Glyph 53668 (\N{HANGUL SYLLABLE TON}) missing from font(s) DejaVu Sans.
  plt.tight_layout(); plt.show()
/tmp/ipython-input-1249101656.py:8: UserWarning: Glyph 50864 (\N{HANGUL SYLLABLE U}) missing from font(s) DejaVu Sans.
  plt.tight_layout(); plt.show()
/tmp/ipython-input-1249101656.py:8: UserWarning: Glyph 51652 (\N{HANGUL SYLLABLE JIN}) missing from font(s) DejaVu Sans.
  plt.tight_layout(); plt.show()
/tmp/ipython-input-1249101656.py:8: UserWarning: Glyph 54840 (\N{HANGUL SYLLABLE HO}) missing from font(s) DejaVu Sans.
  plt.tight_layout(); plt.show()
/tmp/ipython-input-1249101656.py:8: UserWarning: Glyph 44592 (\N{HANGUL SYLLABLE GI}) missing from font(s) DejaVu Sans.
  plt.tight_layout(); plt.show()
/usr/local/lib/python3.12/dist-packages/IPython/core/pylabtools.py:151: UserWarning: Glyph 53668 (\N{HANGUL SYLLABLE TON}) missing from font(s) DejaVu Sans.
  fig.canvas.print_figure(bytes_io, **kw)
/usr/local/lib/python3.12/dist-pack

[Historical plot output omitted; rerun this cell to display it.]

Remark: Density and spread differ between machines — key for quality optimization.


In [ ]:
# V7) Scatter — Max_Injection_Pressure vs Cycle_Time (or fallback) for first equipment
xcol = "Max_Injection_Pressure" if "Max_Injection_Pressure" in df.columns else metricC
ycol = "Cycle_Time" if "Cycle_Time" in df.columns else metricA
m = top_equips[0]
sub = df.loc[df[equip_col]==m, [xcol, ycol]].dropna()
if len(sub)>0:
    plt.scatter(sub[xcol], sub[ycol], alpha=0.5)
    plt.title(f"[V7] Scatter {xcol} vs {ycol} — {equip_col}={m}")
    plt.xlabel(xcol); plt.ylabel(ycol)
    plt.tight_layout(); plt.show()
    print("Remark: Relationship between pressure and cycle time shows machine control differences.")


/tmp/ipython-input-2602338595.py:10: UserWarning: Glyph 53668 (\N{HANGUL SYLLABLE TON}) missing from font(s) DejaVu Sans.
  plt.tight_layout(); plt.show()
/tmp/ipython-input-2602338595.py:10: UserWarning: Glyph 50864 (\N{HANGUL SYLLABLE U}) missing from font(s) DejaVu Sans.
  plt.tight_layout(); plt.show()
/tmp/ipython-input-2602338595.py:10: UserWarning: Glyph 51652 (\N{HANGUL SYLLABLE JIN}) missing from font(s) DejaVu Sans.
  plt.tight_layout(); plt.show()
/tmp/ipython-input-2602338595.py:10: UserWarning: Glyph 54840 (\N{HANGUL SYLLABLE HO}) missing from font(s) DejaVu Sans.
  plt.tight_layout(); plt.show()
/tmp/ipython-input-2602338595.py:10: UserWarning: Glyph 44592 (\N{HANGUL SYLLABLE GI}) missing from font(s) DejaVu Sans.
  plt.tight_layout(); plt.show()
/usr/local/lib/python3.12/dist-packages/IPython/core/pylabtools.py:151: UserWarning: Glyph 53668 (\N{HANGUL SYLLABLE TON}) missing from font(s) DejaVu Sans.
  fig.canvas.print_figure(bytes_io, **kw)
/usr/local/lib/python3.12/dist

[Historical plot output omitted; rerun this cell to display it.]

Remark: Relationship between pressure and cycle time shows machine control differences.


In [ ]:
# V8) Scatter — Filling_Time vs Max_Injection_Speed for first equipment
xcol = "Filling_Time" if "Filling_Time" in df.columns else metricB
ycol = "Max_Injection_Speed" if "Max_Injection_Speed" in df.columns else metricD
m = top_equips[0]
sub = df.loc[df[equip_col]==m, [xcol, ycol]].dropna()
if len(sub)>0:
    plt.scatter(sub[xcol], sub[ycol], alpha=0.5)
    plt.title(f"[V8] Scatter {xcol} vs {ycol} — {equip_col}={m}")
    plt.xlabel(xcol); plt.ylabel(ycol)
    plt.tight_layout(); plt.show()
    print("Remark: Speed-time correlation may highlight hydraulic tuning differences.")


/tmp/ipython-input-4247032195.py:10: UserWarning: Glyph 53668 (\N{HANGUL SYLLABLE TON}) missing from font(s) DejaVu Sans.
  plt.tight_layout(); plt.show()
/tmp/ipython-input-4247032195.py:10: UserWarning: Glyph 50864 (\N{HANGUL SYLLABLE U}) missing from font(s) DejaVu Sans.
  plt.tight_layout(); plt.show()
/tmp/ipython-input-4247032195.py:10: UserWarning: Glyph 51652 (\N{HANGUL SYLLABLE JIN}) missing from font(s) DejaVu Sans.
  plt.tight_layout(); plt.show()
/tmp/ipython-input-4247032195.py:10: UserWarning: Glyph 54840 (\N{HANGUL SYLLABLE HO}) missing from font(s) DejaVu Sans.
  plt.tight_layout(); plt.show()
/tmp/ipython-input-4247032195.py:10: UserWarning: Glyph 44592 (\N{HANGUL SYLLABLE GI}) missing from font(s) DejaVu Sans.
  plt.tight_layout(); plt.show()
/usr/local/lib/python3.12/dist-packages/IPython/core/pylabtools.py:151: UserWarning: Glyph 53668 (\N{HANGUL SYLLABLE TON}) missing from font(s) DejaVu Sans.
  fig.canvas.print_figure(bytes_io, **kw)
/usr/local/lib/python3.12/dist

[Historical plot output omitted; rerun this cell to display it.]

Remark: Speed-time correlation may highlight hydraulic tuning differences.


In [ ]:
# V9) Heatmap — mean(metricA) by equipment × Pass/Fail
if target_col:
    pv = pd.pivot_table(df[df[equip_col].isin(top_equips)], values=metricA, index=equip_col, columns=target_col, aggfunc=np.mean)
    if not pv.empty:
        plt.imshow(pv.values, aspect="auto")
        plt.colorbar()
        plt.yticks(range(len(pv.index)), pv.index)
        plt.xticks(range(len(pv.columns)), pv.columns)
        plt.title(f"[V9] Heatmap of mean({metricA}) by {equip_col} × {target_col}")
        plt.tight_layout(); plt.show()
        print("Remark: Visualizes which machines and outcome states have higher mean cycle times.")


/tmp/ipython-input-1068412303.py:3: FutureWarning: The provided callable <function mean at 0x79ba6ad8c220> is currently using DataFrameGroupBy.mean. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "mean" instead.
  pv = pd.pivot_table(df[df[equip_col].isin(top_equips)], values=metricA, index=equip_col, columns=target_col, aggfunc=np.mean)
/tmp/ipython-input-1068412303.py:10: UserWarning: Glyph 50864 (\N{HANGUL SYLLABLE U}) missing from font(s) DejaVu Sans.
  plt.tight_layout(); plt.show()
/tmp/ipython-input-1068412303.py:10: UserWarning: Glyph 51652 (\N{HANGUL SYLLABLE JIN}) missing from font(s) DejaVu Sans.
  plt.tight_layout(); plt.show()
/tmp/ipython-input-1068412303.py:10: UserWarning: Glyph 53668 (\N{HANGUL SYLLABLE TON}) missing from font(s) DejaVu Sans.
  plt.tight_layout(); plt.show()
/tmp/ipython-input-1068412303.py:10: UserWarning: Glyph 54840 (\N{HANGUL SYLLABLE HO}) missing from font(s) DejaVu Sans.
  plt.

[Historical plot output omitted; rerun this cell to display it.]

Remark: Visualizes which machines and outcome states have higher mean cycle times.


In [ ]:
# V10) Bar — mean Cycle_Time per equipment (or metricA fallback)
bar_metric = "Cycle_Time" if "Cycle_Time" in df.columns else metricA
means = df.groupby(equip_col)[bar_metric].mean().sort_values(ascending=False)
means.plot(kind="bar", rot=0)
plt.title(f"[V10] Mean {bar_metric} by {equip_col}")
plt.xlabel(equip_col); plt.ylabel(f"Mean {bar_metric}")
plt.tight_layout(); plt.show()
print("Remark: Machines differ in mean cycle efficiency; may require process balancing.")


/tmp/ipython-input-3957936160.py:7: UserWarning: Glyph 50864 (\N{HANGUL SYLLABLE U}) missing from font(s) DejaVu Sans.
  plt.tight_layout(); plt.show()
/tmp/ipython-input-3957936160.py:7: UserWarning: Glyph 51652 (\N{HANGUL SYLLABLE JIN}) missing from font(s) DejaVu Sans.
  plt.tight_layout(); plt.show()
/tmp/ipython-input-3957936160.py:7: UserWarning: Glyph 53668 (\N{HANGUL SYLLABLE TON}) missing from font(s) DejaVu Sans.
  plt.tight_layout(); plt.show()
/tmp/ipython-input-3957936160.py:7: UserWarning: Glyph 54840 (\N{HANGUL SYLLABLE HO}) missing from font(s) DejaVu Sans.
  plt.tight_layout(); plt.show()
/tmp/ipython-input-3957936160.py:7: UserWarning: Glyph 44592 (\N{HANGUL SYLLABLE GI}) missing from font(s) DejaVu Sans.
  plt.tight_layout(); plt.show()
/usr/local/lib/python3.12/dist-packages/IPython/core/pylabtools.py:151: UserWarning: Glyph 50864 (\N{HANGUL SYLLABLE U}) missing from font(s) DejaVu Sans.
  fig.canvas.print_figure(bytes_io, **kw)
/usr/local/lib/python3.12/dist-packag

[Historical plot output omitted; rerun this cell to display it.]

Remark: Machines differ in mean cycle efficiency; may require process balancing.



**Analysis remark:**  
- The pivot table suggests how pressure setpoints shift across cycle-time regimes for Pass vs. Fail. Consider interaction terms or non-linear models.



## 4. Conclusions & Next Steps

- **Key EDA Takeaways**
  - Some temperature/pressure variables show strong correlation; reduce redundancy via selection or PCA.
  - Potential anomalies in **mold/barrel temps** (zeros) suggest sensor gaps; treat as missing.
  - Cycle- and injection-time tails indicate sporadic slow cycles; monitor as a leading indicator of defects.

- **Next Steps**
  1. **Outliers**: Detect using IQR or z-score; decide to keep, clip, or remove with justification.
  2.**Missing Data**: Identify MCAR/MAR/MNAR patterns and apply sensible fills or deletions. pressure-time interactions.
  3. **Baseline modeling**: logistic regression / tree ensemble with cross-validation.
  4. **Thresholding** and **cost-sensitive metrics** to reflect scrap vs. false alarms.
